# 第四课：Shared LSTM——让模型按顺序阅读交通历史

上一课的 Shared MLP 一次性读取 12 个历史速度。本课改用 LSTM，让模型从旧到新逐步更新记忆。

完成本课后，你应该能解释：

1. hidden state 和 cell state 分别是什么；
2. 为什么要把 `[B,T,N,1]` 变为 `[B*N,T,1]`；
3. Shared LSTM 为什么能在不同节点数的城市之间加载参数；
4. LSTM 为什么仍然不是空间图模型；
5. 为什么只能用 validation 选择 checkpoint，test 只做最终报告。

## 0. 实验规则

训练集负责梯度更新，validation 负责选择 epoch，test 在模型选择完成后只评估一次。即使没有对 test 反向传播，反复查看 test 并据此挑模型也会泄漏信息。

In [ ]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.evaluation.metrics import horizon_metrics, masked_mae  # noqa: E402
from crosscity.models.lstm import SharedLSTM  # noqa: E402

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
print('device:', device)
print('train/val/test:', len(bundle.train), len(bundle.val), len(bundle.test))

## 1. MLP 和 LSTM 的根本差别

MLP 把 12 个时间步当成 12 个并列输入；LSTM 则重复执行状态更新：

$$h_t,c_t=\mathrm{LSTMCell}(x_t,h_{t-1},c_{t-1})$$

- $h_t$：当前对外输出的短期表示；
- $c_t$：沿时间传递的长期记忆；
- 门控：用 sigmoid 决定遗忘、写入和输出多少信息。

这里不要求背门公式。先掌握计算路径：同一组参数在 12 个时间步反复使用，最后用 $h_{12}$ 预测未来 12 步。

In [ ]:
x, y, mask = next(iter(DataLoader(bundle.train, batch_size=4, shuffle=False)))
B, T, N, F = x.shape
sequences = x.permute(0, 2, 1, 3).reshape(B * N, T, F)
print('原始 x:', x.shape, '= [B,T,N,F]')
print('LSTM 输入:', sequences.shape, '= [B*N,T,F]')
print(f'含义：{B} 个窗口 × {N} 个传感器 = {B*N} 条共享参数的序列')

`reshape(B*N, T, F)` 没有混合不同传感器的数据：每个 `(batch, node)` 组合仍是一条独立的 12 步序列。它只让 PyTorch 把节点维当作额外的 batch。节点之间此时没有通信。

In [ ]:
demo_lstm = nn.LSTM(input_size=1, hidden_size=8, batch_first=True)
all_hidden, (last_hidden, last_cell) = demo_lstm(sequences)
print('每一步 hidden:', all_hidden.shape, '= [B*N,T,H]')
print('最后 hidden:', last_hidden.shape, '= [层数,B*N,H]')
print('最后 cell  :', last_cell.shape, '= [层数,B*N,H]')
print('最后一步是否一致:', torch.allclose(all_hidden[:, -1], last_hidden[-1]))

## 2. 阅读项目中的最小 Shared LSTM

预测头 `Linear(H, 12)` 把最后的时间表示一次性映射成未来 12 步。这叫 direct multi-horizon prediction：不是把第 1 步预测再喂回去预测第 2 步，因此不会发生逐步滚动误差，但不同未来步共享同一个历史表示。

In [ ]:
model = SharedLSTM(output_steps=12, hidden_dim=32)
prediction = model(x)
print(model)
print('prediction:', prediction.shape, 'expected:', y.shape)
print('参数量:', sum(p.numel() for p in model.parameters()))
assert prediction.shape == y.shape

In [ ]:
for name, parameter in model.named_parameters():
    print(f'{name:38s}', tuple(parameter.shape))

print('\n注意：没有任何参数形状包含 N=207。')

这正是跨城市迁移的工程基础：PEMS-BAY 有 325 个节点，但 `input_size=1`、hidden size 和输出步数不变，所以 checkpoint 可以加载。可迁移不代表一定有效；效果仍要由受控实验判断。

## 3. 先检查梯度与单 batch 过拟合

这是训练链路的体检，不是泛化实验。若模型连一个固定 batch 都学不动，应先排查形状、mask、学习率和梯度，而不是直接跑完整训练。

In [ ]:
torch.manual_seed(SEED)
tiny_model = SharedLSTM(12, hidden_dim=32).to(device)
tiny_x, tiny_y, tiny_mask = [item.to(device) for item in next(iter(DataLoader(bundle.train, batch_size=8, shuffle=False)))]
optimizer = torch.optim.Adam(tiny_model.parameters(), lr=1e-2)
tiny_losses = []
for step in range(201):
    optimizer.zero_grad()
    loss = masked_mae(tiny_model(tiny_x), tiny_y, tiny_mask)
    loss.backward()
    optimizer.step()
    tiny_losses.append(loss.item())
print('initial loss:', tiny_losses[0])
print('final loss  :', tiny_losses[-1])
assert tiny_losses[-1] < tiny_losses[0]
plt.plot(tiny_losses)
plt.xlabel('step')
plt.ylabel('normalized masked MAE')
plt.title('Fixed-batch overfit check')
plt.show()

## 4. 明确写出训练与验证循环

为方便先学习，默认 `FAST_MODE=True`，只抽取部分窗口并训练较少 epoch。完成流程后切换为 `False` 得到可比较结果。GPU 推荐 FULL；CPU 先跑 FAST。

In [ ]:
FAST_MODE = True  # 完成首次运行后改为 False
BATCH_SIZE = 64
MAX_EPOCHS = 8 if FAST_MODE else 50
PATIENCE = 3 if FAST_MODE else 8

def limited(dataset, maximum):
    return Subset(dataset, range(min(len(dataset), maximum)))

train_set = limited(bundle.train, 2048) if FAST_MODE else bundle.train
val_set = limited(bundle.val, 1024) if FAST_MODE else bundle.val
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
print('实际训练/验证窗口:', len(train_set), len(val_set))

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total, observed = 0.0, 0
    for xb, yb, mb in loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        with torch.set_grad_enabled(training):
            loss = masked_mae(model(xb), yb, mb)
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        count = int(mb.sum())
        total += loss.item() * count
        observed += count
    return total / observed

# 按有效目标数量加权，避免最后一个小 batch 与完整 batch 权重相同。

In [ ]:
torch.manual_seed(SEED)
lstm = SharedLSTM(12, hidden_dim=32).to(device)
optimizer = torch.optim.Adam(lstm.parameters(), lr=1e-3, weight_decay=1e-4)
best_val, best_state, stale = float('inf'), None, 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(lstm, train_loader, optimizer)
    val_loss = run_epoch(lstm, val_loader)
    history.append({'epoch': epoch, 'train': train_loss, 'val': val_loss})
    print(f'epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}')
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in lstm.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale >= PATIENCE:
            print('early stopping')
            break

lstm.load_state_dict(best_state)
history = pd.DataFrame(history)
history.plot(x='epoch', y=['train', 'val'], marker='o', ylabel='normalized masked MAE')
plt.show()
print('已恢复 validation 最优参数，而不是最后一个 epoch。')

训练 loss 继续下降而 validation 停滞，即使 validation 没有明显上升，也已经是过拟合信号：新增拟合没有转化为未见数据上的收益。

## 5. 最终测试：先反标准化，再计算 mph 误差

模型在标准化空间训练。MAE 要变回原始速度单位才有现实含义；MAPE 也必须使用真实速度作为分母。

In [ ]:
@torch.no_grad()
def collect_predictions(model, dataset):
    model.eval()
    predictions, targets, masks = [], [], []
    for xb, yb, mb in DataLoader(dataset, batch_size=64, shuffle=False):
        predictions.append(model(xb.to(device)).cpu())
        targets.append(yb)
        masks.append(mb)
    return torch.cat(predictions), torch.cat(targets), torch.cat(masks)

evaluation_set = limited(bundle.test, 2048) if FAST_MODE else bundle.test
pred_z, target_z, test_mask = collect_predictions(lstm, evaluation_set)
pred_speed = bundle.scaler.inverse_transform(pred_z)
target_speed = bundle.scaler.inverse_transform(target_z)
lstm_metrics = horizon_metrics(pred_speed, target_speed, test_mask)
pd.DataFrame(lstm_metrics).T.round(3)

FAST 结果只检查代码链路，不能与之前的完整 STGCN/MLP 表公平比较。请将 `FAST_MODE=False` 后从第 4 节重新运行到这里，再执行下面的对照表。

In [ ]:
previous_mae = {
    'Historical Average': {'horizon_3': 4.188, 'horizon_6': 4.188, 'horizon_12': 4.189, 'overall': 4.188},
    'Persistence': {'horizon_3': 3.531, 'horizon_6': 4.252, 'horizon_12': 5.429, 'overall': 4.284},
    'Shared MLP': {'horizon_3': 3.147, 'horizon_6': 3.860, 'horizon_12': 5.004, 'overall': 3.888},
    'STGCN': {'horizon_3': 3.286, 'horizon_6': 3.857, 'horizon_12': 4.786, 'overall': 3.887},
}
previous_mae['Shared LSTM (this run)'] = {key: value['mae'] for key, value in lstm_metrics.items()}
comparison = pd.DataFrame(previous_mae).loc[['horizon_3', 'horizon_6', 'horizon_12', 'overall']]
comparison.round(3).style.highlight_min(axis=1, color='lightgreen')

## 6. 实验证明：它不使用图

`adjacency` 是统一模型接口的一部分，但 Shared LSTM 忽略它。改变邻接矩阵，输出应该完全相同。

In [ ]:
lstm.eval()
sample_x = x[:2].to(device)
with torch.no_grad():
    no_graph = lstm(sample_x, None)
    graph_a = lstm(sample_x, torch.eye(N, device=device))
    graph_b = lstm(sample_x, torch.rand(N, N, device=device))
print('None vs identity 最大差:', (no_graph - graph_a).abs().max().item())
print('None vs random   最大差:', (no_graph - graph_b).abs().max().item())
assert torch.equal(no_graph, graph_a) and torch.equal(no_graph, graph_b)

## 7. 节点数无关性：模拟另一个城市

同一个模型直接处理 325 个节点；这验证的是形状兼容性，不是迁移效果。

In [ ]:
fake_pems_bay = torch.randn(2, 12, 325, 1, device=device)
with torch.no_grad():
    fake_prediction = lstm(fake_pems_bay)
print('输入:', fake_pems_bay.shape, '输出:', fake_prediction.shape)
assert fake_prediction.shape == (2, 12, 325)

## 8. 本课验收问题

请不要查答案，先用自己的话回答：

1. 为什么 LSTM 输入要从 `[B,T,N,1]` 变成 `[B*N,T,1]`？这个操作会让节点互相通信吗？
2. `all_hidden[:, -1]` 与 `last_hidden[-1]` 是什么关系？
3. 本项目为什么用最后 hidden 一次预测 12 步，而不是滚动预测？两者有什么取舍？
4. 为什么 Shared LSTM 的 checkpoint 能从 207 节点城市加载到 325 节点城市？
5. 为什么它比 STGCN 更适合作为“时间模式是否可迁移”的对照？
6. validation 不上涨、train 继续下降，算不算过拟合？为什么？
7. 改变 adjacency 后输出完全不变，说明了什么？

请把以下信息带回来：FULL 模式的 initial/final fixed-batch loss、训练/验证曲线现象、四个 horizon 的 LSTM MAE，以及你最不确定的一道验收题。

下一课我们将从图上最小的操作开始：让每个传感器聚合邻居速度，手算并实现图卷积，再回答“空间信息究竟怎样进入神经网络”。